<a href="https://colab.research.google.com/github/Harshal-2006/Fake-News-Detector-AI/blob/main/fake_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
import pickle
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

Load data

In [13]:
true_df = pd.read_csv('/content/True.csv', engine='python', on_bad_lines='skip')
fake_df = pd.read_csv('/content/Fake.csv')

Assign Label

In [14]:
true_df['label'] = 1  # Real
fake_df['label'] = 0  # Fake

Combine and Shuffle

In [16]:
df = pd.concat([true_df, fake_df]).reset_index(drop=True)
df = df.sample(frac=1).reset_index(drop=True)

Simple Text Cleaning Function

In [17]:
def clean_text(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

df['text'] = df['text'].apply(clean_text)

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\S'
<>:9: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_431/1243301937.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/tmp/ipykernel_431/1243301937.py:5: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
/tmp/ipykernel_431/1243301937.py:9: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


Split Data

In [18]:
x = df['text']
y = df['label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25)

Vectorization (Turning words into numbers)

In [19]:
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)

Model Training

In [22]:
LR = LogisticRegression()
LR.fit(xv_train, y_train)

LogisticRegression()

Checking model's accuracy

In [23]:
print(f"Model Accuracy: {LR.score(xv_test, y_test) * 100:.2f}%")

Model Accuracy: 98.72%


In [24]:
pickle.dump(LR, open('model.pkl', 'wb'))
pickle.dump(vectorization, open('vectorizer.pkl', 'wb'))
print("Files saved! Download 'model.pkl' and 'vectorizer.pkl' from the sidebar.")

Files saved! Download 'model.pkl' and 'vectorizer.pkl' from the sidebar.


In [27]:
def manual_testing(news):

    testing_news = {"text":[news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(clean_text)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    prediction = LR.predict(new_xv_test)

    return "Real News" if prediction[0] == 1 else "Fake News"

# Test it here
news_temp = input("Paste a headline here to test: ")
print(manual_testing(news_temp))

Paste a headline here to test: aliens landed on earth
Fake News
